# 🧬 EVEZ Self-Training Pipeline
### Kaggle T4 GPU Fine-Tuning
**Steps:** Settings → Accelerator → GPU T4 → Run All
**Time:** ~20 min | **Cost:** $0 (free tier)

In [ ]:
!pip install -q transformers datasets peft trl torch accelerate
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')

In [ ]:
import json, urllib.request
url = 'https://raw.githubusercontent.com/EVEZX/neuros/main/training/evez-alpaca.json'
urllib.request.urlretrieve(url, 'evez-alpaca.json')
with open('evez-alpaca.json') as f:
    data = json.load(f)
print(f'✅ Loaded {len(data)} instruction pairs')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

formatted = []
for d in data:
    text = f"### Instruction:\n{d['instruction']}\n### Input:\n{d.get('input','')}\n### Response:\n{d['output']}"
    formatted.append({'text': text})
dataset = Dataset.from_list(formatted)

model_name = 'HuggingFaceTB/SmolLM2-135M'
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map='auto')

lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=['q_proj','v_proj'], task_type='CAUSAL_LM')
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
training_args = SFTConfig(output_dir='./evez-adapter', num_train_epochs=3, per_device_train_batch_size=4, learning_rate=2e-4, logging_steps=10, max_seq_length=512, dataset_text_field='text')
trainer = SFTTrainer(model=model, args=training_args, train_dataset=dataset, processing_class=tokenizer)
trainer.train()

In [ ]:
model.save_pretrained('./evez-adapter')
tokenizer.save_pretrained('./evez-adapter')
print('✅ Training complete! Adapter saved to ./evez-adapter/')